In [ ]:
import sys
sys.path.append('..')

In [ ]:
import copy
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import importlib.util
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, ConfusionMatrixDisplay

from src.dataset import create_train_val_datasets
from config import TRAIN_SIZE, MAX_K, SPLIT_SEED, MAX_SAMPLES_TRAIN, MAX_SAMPLES_VAL, huge_dataset_path, SPLIT_STRATEGY, SEED_TRAIN, SEED_VAL

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device : {device}") 

learning_strategy = "ordinal"
run_name = Path("runs/run_20260608_153738")
run_path = "../" / run_name / "checkpoints"
model_structure_path = run_path / "model_structure.py"
ckpt_path = run_path / "best_checkpoint.pth"
dataset_path = Path("/sps/l2it/tdonze/gb-dataset-gen/data/synthetic_dataset/dataset_500K_2.0e-23_1.0e-22_filtering_True_10_100_difficulty_1/dataset.hdf5")

In [ ]:
spec = importlib.util.spec_from_file_location(
    "dynamic_model", model_structure_path
)
dynamic_model_module = importlib.util.module_from_spec(spec)
sys.modules["dynamic_model"] = dynamic_model_module
spec.loader.exec_module(dynamic_model_module)

CardinalityEstimator = getattr(
    dynamic_model_module, "CardinalityEstimator"
)

In [ ]:
model = CardinalityEstimator(learning_strategy="ordinal", max_K=MAX_K, input_channels=4).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

In [ ]:


_, base_ds, _ = create_train_val_datasets(
    dataset_path=huge_dataset_path,
    train_size=TRAIN_SIZE,
    max_K=MAX_K,
    max_samples_val=None,
    noise_val=True,
    deterministic_val=True,
    split_seed=SPLIT_SEED,
    split_strategy=SPLIT_STRATEGY,
    seed_train=SEED_TRAIN,
    seed_val=SEED_VAL
)

In [ ]:
def make_dataset_view(base_ds, seed, noise):
    ds = copy.copy(base_ds)

    ds.return_params = True
    ds.seed = seed
    ds.noise = noise
    ds.rng = np.random.default_rng(seed)
    
    ds.fixed_mixtures = ds._build_fixed_mixtures()

    return ds

In [ ]:
# NB_RUNS = 10
# seeds = [0, 2026, 45, 1234, 5678, 91011, 121314, 151617, 181920, 212223]
NB_RUNS = 1
seeds = [0]
assert NB_RUNS == len(seeds), "NB_RUNS should be equal to len(seeds)"

all_results = []

done = False
path = Path("ordinal_evaluation_results.csv")


if path.exists():
    print(f"File {path} already exists. Loading existing results.")
    results_df = pd.read_csv(path)
    done = True

if not done :
    for run in range(NB_RUNS):
        seed = seeds[run]

        print(f"Copying dataset for run {run + 1}, seed : {seed}")
        val_ds = make_dataset_view(base_ds, seed=seed, noise=True)

        signal_ds = copy.copy(val_ds)
        signal_ds.noise = False
        print(f"Dataset copied for run {run + 1}")

        val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
        signal_loader = DataLoader(signal_ds, batch_size=512, shuffle=False)

        rows = []
        offset = 0

        with torch.no_grad():
            progress = tqdm(
                zip(val_loader, signal_loader),
                total=len(val_loader),
                desc=f"Run {run + 1}/{NB_RUNS}",
                leave=True,
            )

            for (summed_waveform, targets, params), (signal_waveform, signal_targets, signal_params) in progress:
                X = torch.as_tensor(summed_waveform, dtype=torch.float32, device=device)
                X_signal = torch.as_tensor(signal_waveform, dtype=torch.float32, device=device)

                targets = torch.as_tensor(targets, dtype=torch.long, device=device)
                signal_targets = torch.as_tensor(signal_targets, dtype=torch.long, device=device)

                assert torch.equal(targets, signal_targets)

                logits = model(X)

                if learning_strategy == "mse":
                    y = targets.float().unsqueeze(1) / MAX_K # shape : (batch_size, 1)
                    out = torch.sigmoid(logits) # shape : (batch_size, 1)

                    pred_score = out.squeeze(1) * MAX_K # shape : (batch_size,)
                    pred = torch.round(pred_score).long().clamp(1, MAX_K) # shape : (batch_size,)

                    loss_per_sample = F.mse_loss(out, y, reduction="none").squeeze(1) # shape : (batch_size,)
                elif learning_strategy == "ordinal":
                    thresholds = torch.arange(1, MAX_K, device=device) # shape3 : (max_K - 1,)
                    y = (targets.unsqueeze(1) > thresholds.unsqueeze(0)).float() # shape : (batch_size, max_K - 1)

                    probs = torch.sigmoid(logits)

                    pred = 1 + (probs > 0.5).sum(dim=1)
                    pred = pred.clamp(1, MAX_K)

                    pred_score = 1.0 + probs.sum(dim=1)

                    loss_per_sample = F.binary_cross_entropy_with_logits(
                        logits,
                        y,
                        reduction="none",
                    ).mean(dim=1) 
                elif learning_strategy == "cross_entropy":
                    y = targets - 1

                    probs = F.softmax(logits, dim=1)
                    class_values = torch.arange(1, MAX_K + 1, device=device).float()

                    pred = logits.argmax(dim=1) + 1
                    pred_score = (probs * class_values.unsqueeze(0)).sum(dim=1)

                    loss_per_sample = F.cross_entropy(logits, y, reduction="none")
                else:
                    raise ValueError(f"Unknown learning strategy : {learning_strategy}")
                
                snr = torch.as_tensor(params["snr"], dtype=torch.float32, device=device)
                source_mask = torch.as_tensor(params["source_mask"], dtype=torch.bool, device=device)

                snr_masked = torch.where(source_mask, snr, torch.zeros_like(snr))
                snr_mix = torch.sqrt((snr_masked ** 2).sum(dim=1))
                max_snr = snr_masked.max(dim=1).values
                mean_snr = snr_masked.sum(dim=1) / source_mask.sum(dim=1).clamp(min=1)

                n_visible_5 = ((snr > 5) & source_mask).sum(dim=1)
                n_visible_10 = ((snr > 10) & source_mask).sum(dim=1)
                n_visible_20 = ((snr > 20) & source_mask).sum(dim=1)

                energy_noisy = (X ** 2).sum(dim=(1, 2))
                energy_signal = (X_signal ** 2).sum(dim=(1, 2))

                batch_size = targets.shape[0]

                for i in range(batch_size):
                    valid_snr = snr[i][source_mask[i]].detach().cpu().numpy()

                    rows.append({
                        "run": run,
                        "seed": seed,
                        "idx": offset + i,

                        "true_K": int(targets[i].item()),
                        "pred_K": int(pred[i].item()),
                        "pred_score": float(pred_score[i].item()),

                        "correct": bool(pred[i].item() == targets[i].item()),
                        "error_K": int(pred[i].item() - targets[i].item()),
                        "abs_error_K": int(abs(pred[i].item() - targets[i].item())),

                        "loss": float(loss_per_sample[i].item()),

                        "snr_values": valid_snr.tolist(),
                        "snr_mix": float(snr_mix[i].item()),
                        "max_snr": float(max_snr[i].item()),
                        "mean_snr": float(mean_snr[i].item()),

                        "n_visible_5": int(n_visible_5[i].item()), # correspond to the number of sources with SNR > 5
                        "n_visible_10": int(n_visible_10[i].item()), # correspond to the number of sources with SNR > 10
                        "n_visible_20": int(n_visible_20[i].item()), # correspond to the number of sources with SNR > 20

                        "energy_noisy": float(energy_noisy[i].item()),
                        "energy_signal": float(energy_signal[i].item()),
                    })

                offset += batch_size

        df = pd.DataFrame(rows)
        all_results.append(df)
    results_df = pd.concat(all_results, ignore_index=True)


In [ ]:
print(f"Saving results to {path}")
results_df.to_csv(path, index=False)

In [ ]:
LABELS = np.arange(1, MAX_K + 1)

def compute_metrics(df, target_col="true_K", pred_col="pred_K"):
    y_true = df[target_col].to_numpy()
    y_pred = df[pred_col].to_numpy()

    return pd.Series({
        "n": len(df),
        "acc": accuracy_score(y_true, y_pred),
        "acc_pm1": np.mean(np.abs(y_true - y_pred) <= 1),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "mae": mean_absolute_error(y_true, y_pred),
        "mean_error": np.mean(y_pred - y_true),
    })

#### Global Metrics Calculation and Display

In [ ]:
metrics_by_run = results_df.groupby("run").apply(compute_metrics)
print("Metrics by run :")
display(metrics_by_run)
print("Mean metrics across runs :")
display(metrics_by_run.mean().rename("mean"))
print("Standard deviation of metrics across runs :")
display(metrics_by_run.std().rename("std"))

In [ ]:
cm = confusion_matrix(results_df["true_K"], results_df["pred_K"], labels=LABELS)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

ax.set_title("Normalized confusion matrix: true_K vs pred_K")
ax.set_xlabel("Predicted K")
ax.set_ylabel("True K")
ax.set_xticks(np.arange(len(LABELS)))
ax.set_yticks(np.arange(len(LABELS)))
ax.set_xticklabels(LABELS)
ax.set_yticklabels(LABELS)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(
            j, i, f"{cm_norm[i, j]:.2f}",
            ha="center", va="center",
            fontsize=8,
            color="white" if cm_norm[i, j] > cm_norm.max() / 2 else "black",
        )

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()